# 08 - Benchmark, Evaluation and Consolidated Results
## Objective 9, the evaluation strategy, and the final result set

**Ahsanullah University of Science and Technology** - Department of Computer Science and Engineering

**Course:** CSE 4262 Data Analytics Lab | **Lab Group:** Gr-03 | **Group:** Gr-06

| Student ID | Name |
|---|---|
| 20220104006 | A.S.M. Tahsin Tajware |
| 20220104014 | Abdullah Al Tamim |
| 20220104032 | Eusha Ahmed Mahi |


### Purpose

**Objective 9** times four operations in both engines at three data sizes. Spark is measured twice:
cold, where the CSV is re-read on every run, and cached, where the data is resident in memory.
Each timing is repeated and reported as a mean with its standard deviation, because a single run
on a JVM says very little.

The evaluation section then audits correctness against pandas and checks the findings against the
literature, and the final section assembles the headline result from every notebook in the
pipeline by reading the tables they wrote.

In [ ]:
import os, sys, glob

_candidates = ["/kaggle/working/repo", "/kaggle/working", "..", "."] + [
    os.path.dirname(p) for p in glob.glob("/kaggle/input/**/da_common.py", recursive=True)]
for _p in _candidates:
    if os.path.exists(os.path.join(_p, "da_common.py")):
        sys.path.insert(0, os.path.abspath(_p))
        break
else:
    raise FileNotFoundError("da_common.py not found. See KAGGLE_SETUP.md for the two setup options.")

from da_common import *

banner("Notebook 08 - Objective 9 and evaluation")
spark = get_spark("08 benchmark and evaluation")
df = load_analytical(spark).cache()
n_clean = df.count()
print(f"analytical dataset: {n_clean:,} rows")

### 1. Benchmark harness

In [ ]:
import gc

print("Loading the same file into pandas...")
t0 = time.perf_counter()
pdf_full = pd.read_csv(DATA_PATH)
t_load = time.perf_counter() - t0
print(f"pandas rows: {len(pdf_full):,} (read_csv took {t_load:.2f} s)")
print(f"pandas memory: {pdf_full.memory_usage(deep=True).sum()/1e6:,.0f} MB")

REPEATS_WARM, REPEATS_COLD = 5, 3
FRACTIONS = [0.10, 0.50, 1.00]
OPS = ["groupby_agg", "multi_agg", "filter_count", "sort_take"]

def spark_ops(sdf):
    return {
        "groupby_agg":  lambda: sdf.groupBy("Category").agg(F.avg("rating"), F.count("*")).collect(),
        "multi_agg":    lambda: sdf.groupBy("Category", "rating")
                                   .agg(F.avg("helpful_vote"), F.count("*"), F.max("helpful_vote")).collect(),
        "filter_count": lambda: sdf.filter((F.col("rating") == 5) &
                                           (F.col("verified_purchase") == True)).count(),
        "sort_take":    lambda: sdf.orderBy(F.desc("helpful_vote")).limit(20).collect(),
    }

def pandas_ops(pdf):
    return {
        "groupby_agg":  lambda: pdf.groupby("Category").agg(avg=("rating", "mean"), n=("rating", "size")),
        "multi_agg":    lambda: pdf.groupby(["Category", "rating"]).agg(
                                    a=("helpful_vote", "mean"), n=("helpful_vote", "size"),
                                    m=("helpful_vote", "max")),
        "filter_count": lambda: int(((pdf["rating"] == 5) & (pdf["verified_purchase"] == True)).sum()),
        "sort_take":    lambda: pdf.sort_values("helpful_vote", ascending=False).head(20),
    }

def timed(fn, repeats, warmup=False):
    if warmup:
        fn()
    ts = []
    for _ in range(repeats):
        t = time.perf_counter(); fn(); ts.append(time.perf_counter() - t)
    return float(np.mean(ts)), float(np.std(ts))

print("Harness ready.")

In [ ]:
rows_out = []
for frac in FRACTIONS:
    cold_src = read_raw(spark, tag=f"cold_{int(frac*100)}")
    warm_src = read_raw(spark, tag=f"warm_{int(frac*100)}")
    if frac < 1.0:
        cold_src = cold_src.sample(False, frac, seed=42)
        warm_src = warm_src.sample(False, frac, seed=42)
    warm_src = warm_src.cache()
    n_rows = warm_src.count()
    pdf_s = pdf_full if frac >= 1.0 else pdf_full.sample(frac=frac, random_state=42)

    s_cold, s_warm, p_ops = spark_ops(cold_src), spark_ops(warm_src), pandas_ops(pdf_s)
    for op in OPS:
        mc, sc = timed(s_cold[op], REPEATS_COLD)
        mw, sw = timed(s_warm[op], REPEATS_WARM, warmup=True)
        mp, sp = timed(p_ops[op], REPEATS_WARM, warmup=True)
        rows_out += [(frac, n_rows, op, "PySpark (cold)", mc, sc),
                     (frac, n_rows, op, "PySpark (cached)", mw, sw),
                     (frac, n_rows, op, "Pandas", mp, sp)]
        print(f"size={frac:>4.0%} ({n_rows:>7,})  {op:<13} "
              f"cold={mc*1000:8.1f} ms  cached={mw*1000:7.1f} ms  pandas={mp*1000:7.1f} ms")
    warm_src.unpersist(); gc.collect()

bench = pd.DataFrame(rows_out, columns=["fraction", "rows", "operation", "engine", "mean_sec", "std_sec"])
bench["mean_ms"] = (bench["mean_sec"] * 1000).round(2)
bench["std_ms"] = (bench["std_sec"] * 1000).round(2)

pivot_bench = bench.pivot_table(index=["operation", "rows"], columns="engine",
                                values="mean_ms", aggfunc="first").round(1)
print("\nMean runtime in ms"); print(pivot_bench)
save_table(bench, "obj9_benchmark_raw")
save_table(pivot_bench.reset_index(), "obj9_benchmark_summary");

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))
style = {"PySpark (cold)": ("#e11d48", "o"), "PySpark (cached)": ("#f59e0b", "s"),
         "Pandas": ("#2563eb", "^")}
for ax, op in zip(axes, OPS):
    sub = bench[bench["operation"] == op]
    for eng, (color, mk) in style.items():
        s = sub[sub["engine"] == eng].sort_values("rows")
        ax.errorbar(s["rows"], s["mean_ms"], yerr=s["std_ms"], marker=mk, color=color,
                    lw=2, capsize=3, label=eng)
    ax.set_title(op); ax.set_xlabel("rows"); ax.set_yscale("log")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
axes[0].set_ylabel("time (ms, log scale)"); axes[0].legend(fontsize=8)
plt.suptitle("Objective 9 - PySpark vs Pandas across data sizes (lower is faster)",
             y=1.05, fontsize=13, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig12_benchmark_by_operation", "Objective 9 - runtime by operation and size")
plt.show()

In [ ]:
ratio = (bench[bench["engine"] != "PySpark (cold)"]
         .pivot_table(index=["operation", "rows"], columns="engine", values="mean_sec")
         .assign(ratio=lambda d: d["PySpark (cached)"] / d["Pandas"]).reset_index())

fig, ax = plt.subplots(figsize=(10, 4.6))
for op in OPS:
    s = ratio[ratio["operation"] == op].sort_values("rows")
    ax.plot(s["rows"], s["ratio"], marker="o", lw=2, label=op)
ax.axhline(1.0, color="#64748b", ls="--", lw=1); ax.set_yscale("log")
ax.set_title("Cached PySpark time relative to Pandas (downward slope = Spark closing the gap)")
ax.set_xlabel("rows"); ax.set_ylabel("PySpark time / Pandas time")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax.legend(fontsize=9)
plt.tight_layout()
savefig(fig, "fig13_benchmark_scaling", "Objective 9 - scaling trend of the time ratio")
plt.show()
save_table(ratio, "obj9_scaling_ratio");

### 2. Correctness auditing

Every headline aggregate is recomputed in pandas on the same sampled subset and compared with the
Spark result. A mismatch would mean the Spark query is not asking what it appears to ask.

In [ ]:
audit_sample = df.select("Category", "rating", "verified_purchase", "helpful_vote",
                        "review_length", "user_id", "review_year").sample(False, 0.05, seed=11).cache()
a_pdf = audit_sample.toPandas()
print(f"Audit sample: {len(a_pdf):,} rows")

checks = []
def check(name, sv, pv, tol=1e-6):
    ok = abs(float(sv) - float(pv)) <= tol
    checks.append((name, round(float(sv), 6), round(float(pv), 6), "PASS" if ok else "FAIL"))

check("row count", audit_sample.count(), len(a_pdf))
check("mean rating", audit_sample.agg(F.avg("rating")).first()[0], a_pdf["rating"].mean(), 1e-9)
check("verified share", audit_sample.agg(F.avg(F.col("verified_purchase").cast("int"))).first()[0],
      a_pdf["verified_purchase"].mean(), 1e-9)
check("mean helpful votes", audit_sample.agg(F.avg("helpful_vote")).first()[0],
      a_pdf["helpful_vote"].mean(), 1e-9)
check("mean review length", audit_sample.agg(F.avg("review_length")).first()[0],
      a_pdf["review_length"].mean(), 1e-9)
check("count of 5-star", audit_sample.filter(F.col("rating") == 5).count(),
      int((a_pdf["rating"] == 5).sum()))
check("distinct reviewers", audit_sample.select("user_id").distinct().count(), a_pdf["user_id"].nunique())
check("max helpful votes", audit_sample.agg(F.max("helpful_vote")).first()[0], a_pdf["helpful_vote"].max())

s_cat = (audit_sample.groupBy("Category").agg(F.avg("rating").alias("m"))
         .toPandas().set_index("Category")["m"].sort_index())
p_cat = a_pdf.groupby("Category")["rating"].mean().sort_index()
check("max per-category mean gap", float((s_cat - p_cat).abs().max()), 0.0, 1e-9)

audit_pdf = pd.DataFrame(checks, columns=["check", "spark", "pandas", "result"])
print(audit_pdf.to_string(index=False))
print("\nAll correctness checks passed." if (audit_pdf["result"] == "PASS").all()
      else "\nSome checks FAILED - investigate above.")
save_table(audit_pdf, "eval_correctness_audit")
audit_sample.unpersist();

### 3. External consistency

In [ ]:
rating_tbl = load_table("obj1_rating_distribution")
ver_tbl = load_table("obj2_verified_comparison")
len_tbl = load_table("obj7_length_by_rating")

rows = []
if rating_tbl is not None:
    five = float(rating_tbl.loc[rating_tbl["rating"] == 5.0, "pct"].iloc[0])
    rows.append(("Positivity skew (Chevalier and Mayzlin, 2006)", f"{five}% of reviews are 5-star",
                 "Consistent with the reported positivity bias in online reviews"))
if ver_tbl is not None:
    v = ver_tbl[ver_tbl["verified_purchase"] == True].iloc[0]
    nv = ver_tbl[ver_tbl["verified_purchase"] == False].iloc[0]
    rows.append(("Verified-purchase share", f"{v['share_pct']}% verified",
                 "High badge coverage, so the badge alone separates very little"))
    rows.append(("Verified vs non-verified gap (He et al., 2022)",
                 f"rating {v['avg_rating']-nv['avg_rating']:+.3f}, "
                 f"length {v['avg_review_length']-nv['avg_review_length']:+.0f} chars",
                 "Reported descriptively, not treated as evidence of manipulation"))
if len_tbl is not None:
    gap = (float(len_tbl.loc[len_tbl["rating"] == 5.0, "avg_chars"].iloc[0]) -
           float(len_tbl.loc[len_tbl["rating"] == 1.0, "avg_chars"].iloc[0]))
    rows.append(("Length against rating", f"5-star run {gap:+.0f} chars vs 1-star",
                 "Satisfied customers write less, dissatisfied ones explain at length"))

consistency = pd.DataFrame(rows, columns=["Reference point", "Measured", "Reading"])
save_table(consistency, "eval_external_consistency")
consistency

### 4. Consolidated findings

Assembled by reading the tables the earlier notebooks wrote, so this section stays correct when
any upstream notebook is re-run.

In [ ]:
def pick(name, col, row=0, default=None):
    t = load_table(name)
    if t is None or col not in t.columns or len(t) <= row:
        return default
    return t[col].iloc[row]

sat = load_table("obj1_satisfaction_by_category")
findings = []

if rating_tbl is not None:
    avg_all = (rating_tbl["rating"] * rating_tbl["count"]).sum() / rating_tbl["count"].sum()
    line = f"Average {avg_all:.3f} stars, {rating_tbl.loc[rating_tbl['rating']==5.0,'pct'].iloc[0]}% five-star"
    if sat is not None:
        hi, lo = sat.iloc[sat["avg_rating"].idxmax()], sat.iloc[sat["avg_rating"].idxmin()]
        line += f"; {hi['Category']} highest at {hi['avg_rating']}, {lo['Category']} lowest at {lo['avg_rating']}"
    findings.append(("1. Customer satisfaction", line))

if ver_tbl is not None:
    findings.append(("2. Verified purchase",
        f"{v['share_pct']}% verified; non-verified reviews are "
        f"{nv['avg_review_length']/v['avg_review_length']:.1f}x longer and receive "
        f"{nv['avg_helpful_vote']/max(v['avg_helpful_vote'],1e-9):.1f}x the helpful votes"))

if pick("obj3_helpful_summary", "mean_helpful_vote") is not None:
    findings.append(("3. Helpful reviews",
        f"Mean {pick('obj3_helpful_summary','mean_helpful_vote')} votes, only "
        f"{pick('obj3_helpful_summary','pct_with_any_vote')}% of reviews receive any"))

if pick("obj4_trend_summary", "peak_year") is not None:
    findings.append(("4. Temporal trend",
        f"Peak year {int(pick('obj4_trend_summary','peak_year'))} with "
        f"{int(pick('obj4_trend_summary','peak_reviews')):,} reviews; 2019-2021 was "
        f"{pick('obj4_trend_summary','surge_multiple')}x the 2016-2018 volume"))

if pick("obj5_product_summary", "products") is not None:
    findings.append(("5. Product analytics",
        f"{int(pick('obj5_product_summary','eligible_products')):,} of "
        f"{int(pick('obj5_product_summary','products')):,} products clear the "
        f"{int(pick('obj5_product_summary','min_reviews'))}-review threshold"))

act_tbl = load_table("obj6_activity_distribution")
if act_tbl is not None and pick("obj6_reviewer_summary", "reviewers") is not None:
    one = act_tbl.loc[act_tbl["bucket"].astype(str) == "1", "pct_reviewers"]
    findings.append(("6. Customer activity",
        f"{int(pick('obj6_reviewer_summary','reviewers')):,} reviewers, "
        f"{one.iloc[0] if len(one) else 'n/a'}% wrote exactly one review"))

if len_tbl is not None:
    findings.append(("7. Text analytics",
        f"5-star reviews run {gap:+.0f} chars vs 1-star; vocabulary of "
        f"{int(pick('obj7_vocab_summary','vocab_size', default=0)):,} terms at minDF=5"))

findings.append(("8. Window functions",
    f"dense_rank per category, row_number over reviewers, trailing-{ROLLING_WINDOW} rolling average"))

last = ratio.sort_values("rows").iloc[-1]
findings.append(("9. Benchmark",
    f"At the full size, cached PySpark takes {last['ratio']:.1f}x the Pandas time on {last['operation']}"))

final = pd.DataFrame(findings, columns=["Objective", "Headline result"])
save_table(final, "final_consolidated_findings")
final

### Conclusion

The pipeline runs end to end: a schema-declared load, a profile taken before cleaning, a logged
four-step preprocessing pass, features derived once into a persisted Parquet dataset, and nine
objectives answered from that single source.

Three results carry the report. Ratings are J-shaped rather than bell-shaped, so an average rating
hides the distribution and product ranking needs a review threshold to mean anything. Helpful votes
reach a small minority of reviews and are confounded with review age, so helpfulness is a rare
event rather than a continuous score. And the benchmark behaves the way the Spark literature
predicts: a fixed scheduling and JVM overhead dominates at small sizes, and the relative cost falls
as the data grows. That is the argument for using Spark here, measured rather than assumed.

Every objective is backed by a saved table and, where it helps, a saved figure, so the written
report can be assembled directly from the `results/` and `figures/` folders.


In [ ]:
print(f"Result tables in {TBL_DIR}")
for f in sorted(os.listdir(TBL_DIR)):
    if f.endswith(".csv"):
        print("  ", f)
print(f"\nFigures in {FIG_DIR}")
for f in sorted(os.listdir(FIG_DIR)):
    print("  ", f)

In [ ]:
spark.stop()
print("Spark session stopped.")